In [ ]:
# 场景 1：首次建库（一键初始化，再次运行，已有值会跳过）
from market_data import load_tushare_token
from stock_data import create_stock_manager

TUSHARE_TOKEN = load_tushare_token()

manager = create_stock_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path="data/db/stock_data.db",
    default_start_date="20130101",  # 可按需要改成更早，例如 20100101
    default_exchange="SSE",
    retry_times=3,
    retry_sleep=1.0,
)

summary = manager.initialize_database(
    start_date="20130101",
    include_factor_pro=True,
    include_moneyflow=True,
    include_st_daily=True,
    build_eligibility=True,
    show_progress=True,
)

summary

In [ ]:
# 场景 2：日常维护（自动补齐到最新交易日）
from datetime import datetime, timedelta

from market_data import load_tushare_token
from stock_data import create_stock_manager

TUSHARE_TOKEN = load_tushare_token()

manager = create_stock_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path="data/db/stock_data.db",
    default_start_date="20130101",
    default_exchange="SSE",
)

# 1) 先把交易日历刷新到最新
# 用最近一段时间回补即可，不需要每次都从 2013 开始拉
calendar_start = (datetime.now() - timedelta(days=120)).strftime("%Y%m%d")
manager.initialize_trade_calendar(
    start_date=calendar_start,
    end_date=None,
    exchange=manager.config.default_exchange,
)

open_trade_dates = manager.store.get_open_trade_dates(
    exchange=manager.config.default_exchange
)

if not open_trade_dates:
    raise RuntimeError("交易日历为空，无法继续维护。")

latest_market_trade_date = open_trade_dates[-1]

def next_missing_tail_start(table_name: str):
    latest_db_trade_date = manager.store.get_latest_trade_date(table_name)

    # 表还是空的：从默认起点开始首建
    if latest_db_trade_date is None:
        return manager.config.default_start_date, latest_db_trade_date

    # 找数据库最新日之后的第一个交易日
    missing_tail = [d for d in open_trade_dates if d > latest_db_trade_date]
    if not missing_tail:
        return None, latest_db_trade_date

    return missing_tail[0], latest_db_trade_date

summary = {
    "latest_market_trade_date": latest_market_trade_date,
    "tables": {},
}

jobs = [
    ("stock_daily_raw", manager.backfill_stock_daily),
    ("stock_adj_factor", manager.backfill_stock_adj_factor),
    ("stock_daily_basic", manager.backfill_stock_daily_basic),
    ("stock_moneyflow", manager.backfill_stock_moneyflow),
    ("stock_st_daily", manager.backfill_stock_st_daily),
    ("stock_factor_pro", manager.backfill_stock_factor_pro),
]

eligibility_rebuild_starts = []

for table_name, backfill_func in jobs:
    start_date, latest_db_trade_date = next_missing_tail_start(table_name)

    if start_date is None:
        summary["tables"][table_name] = {
            "status": "up_to_date",
            "latest_db_trade_date": latest_db_trade_date,
            "latest_market_trade_date": latest_market_trade_date,
        }
        continue

    result = backfill_func(
        start_date=start_date,
        end_date=latest_market_trade_date,
        show_progress=True,
        skip_existing=True,
    )

    summary["tables"][table_name] = {
        "status": "updated",
        "latest_db_trade_date_before_update": latest_db_trade_date,
        "update_start_date": start_date,
        "update_end_date": latest_market_trade_date,
        "result": result,
    }

    # eligibility 主要依赖这些表
    if table_name in {"stock_daily_raw", "stock_daily_basic", "stock_moneyflow", "stock_st_daily"}:
        eligibility_rebuild_starts.append(start_date)

# 2) 如有需要，顺手重建 eligibility 对应区间
if eligibility_rebuild_starts:
    eligibility_start = min(eligibility_rebuild_starts)

    eligibility_result = manager.build_eligibility_daily(
        start_date=eligibility_start,
        end_date=latest_market_trade_date,
        min_list_days=60,
        min_amount=20000,
        min_total_mv=None,
        max_total_mv=3000000,
        require_daily_basic=True,
        require_moneyflow=False,
        exclude_st=True,
        show_progress=True,
        skip_existing=False,   # 这里建议重建对应区间
    )

    summary["stock_eligibility_daily"] = {
        "status": "rebuilt",
        "start_date": eligibility_start,
        "end_date": latest_market_trade_date,
        "result": eligibility_result,
    }
else:
    summary["stock_eligibility_daily"] = {
        "status": "skipped",
        "message": "上游依赖表没有新增交易日，无需重建",
    }

summary

In [ ]:
# 场景 3：按表分步回补（适合中断后补跑或单独补某张表）
from market_data import load_tushare_token
from stock_data import create_stock_manager

TUSHARE_TOKEN = load_tushare_token()

manager = create_stock_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path="data/db/stock_data.db",
    default_start_date="20130101",
    default_exchange="SSE",
)

daily_summary = manager.backfill_stock_daily(
    start_date="20230101",
    end_date="20231231",
    show_progress=True,
    skip_existing=True,
)

adj_summary = manager.backfill_stock_adj_factor(
    start_date="20230101",
    end_date="20231231",
    show_progress=True,
    skip_existing=True,
)

basic_summary = manager.backfill_stock_daily_basic(
    start_date="20230101",
    end_date="20231231",
    show_progress=True,
    skip_existing=True,
)

moneyflow_summary = manager.backfill_stock_moneyflow(
    start_date="20230101",
    end_date="20231231",
    show_progress=True,
    skip_existing=True,
)

st_summary = manager.backfill_stock_st_daily(
    start_date="20230101",
    end_date="20231231",
    show_progress=True,
    skip_existing=True,
)

factor_pro_summary = manager.backfill_stock_factor_pro(
    start_date="20230101",
    end_date="20231231",
    show_progress=True,
    skip_existing=True,
)

{
    "daily": daily_summary,
    "adj_factor": adj_summary,
    "daily_basic": basic_summary,
    "moneyflow": moneyflow_summary,
    "stock_st_daily": st_summary,
    "factor_pro": factor_pro_summary,
}

In [ ]:
# 场景 4：重建 / 更新 eligibility 日表（动态资产池过滤辅助表）
from market_data import load_tushare_token
from stock_data import create_stock_manager

TUSHARE_TOKEN = load_tushare_token()

manager = create_stock_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path="data/db/stock_data.db",
    default_start_date="20130101",
    default_exchange="SSE",
)

eligibility_summary = manager.build_eligibility_daily(
    start_date="20240101",
    end_date="20241231",
    min_list_days=60,
    min_amount=20000,       # 成交额下限（单位跟原表一致）
    min_total_mv=None,
    max_total_mv=3000000,   # 例：限制总市值上限，便于做中小盘/微盘研究
    require_daily_basic=True,
    require_moneyflow=False,
    exclude_st=True,
    show_progress=True,
    skip_existing=False,
)

eligibility_summary

In [ ]:
# 场景 5：读取某日宽截面快照（最常用研究入口）
from market_data import load_tushare_token
from stock_data import create_stock_manager

TUSHARE_TOKEN = load_tushare_token()

manager = create_stock_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path="data/db/stock_data.db",
    default_start_date="20130101",
    default_exchange="SSE",
)

snapshot = manager.get_cross_section_snapshot(
    trade_date="20241231",
    include_daily=True,
    include_basic=True,
    include_moneyflow=True,
    include_factor_pro=True,
    include_st=True,
    include_eligibility=True,
    adjust_type="qfq",
    eligible_only=True,
    min_total_mv=None,
    max_total_mv=3000000,
)

print(snapshot.shape)
snapshot.head()

In [ ]:
# 场景 6：读取价格（raw / qfq / hfq），适合做标签或检查单股时间序列
from market_data import load_tushare_token
from stock_data import create_stock_manager

TUSHARE_TOKEN = load_tushare_token()

manager = create_stock_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path="data/db/stock_data.db",
    default_start_date="20130101",
    default_exchange="SSE",
)

prices_qfq = manager.get_prices(
    ts_codes=["000001.SZ", "300750.SZ", "600519.SH"],
    start_date="20240101",
    end_date="20241231",
    adjust_type="qfq",
)

prices_raw = manager.get_prices(
    ts_codes=["000001.SZ", "300750.SZ", "600519.SH"],
    start_date="20240101",
    end_date="20241231",
    adjust_type="raw",
)

print("qfq:", prices_qfq.shape)
print("raw:", prices_raw.shape)

prices_qfq.head()

In [ ]:
# 场景 7：直接读取库中的几张关键表，做检查或自定义拼接
from market_data import load_tushare_token
from stock_data import create_stock_manager

TUSHARE_TOKEN = load_tushare_token()

manager = create_stock_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path="data/db/stock_data.db",
    default_start_date="20130101",
    default_exchange="SSE",
)

daily_df = manager.store.get_stock_daily_raw(
    start_date="20241201",
    end_date="20241231",
)

basic_df = manager.store.get_stock_daily_basic(
    start_date="20241201",
    end_date="20241231",
)

moneyflow_df = manager.store.get_stock_moneyflow(
    start_date="20241201",
    end_date="20241231",
)

factor_pro_df = manager.store.get_stock_factor_pro(
    start_date="20241201",
    end_date="20241231",
    expand_json=False,  # True 时会把 data_json 展开成列
)

eligibility_df = manager.store.get_stock_eligibility_daily(
    start_date="20241201",
    end_date="20241231",
)

{
    "daily_shape": daily_df.shape,
    "basic_shape": basic_df.shape,
    "moneyflow_shape": moneyflow_df.shape,
    "factor_pro_shape": factor_pro_df.shape,
    "eligibility_shape": eligibility_df.shape,
}

In [ ]:
# 场景 8：查看更新日志、检查哪些交易日已经入库
from market_data import load_tushare_token
from stock_data import create_stock_manager

TUSHARE_TOKEN = load_tushare_token()

manager = create_stock_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path="data/db/stock_data.db",
    default_start_date="20130101",
    default_exchange="SSE",
)

log_df = manager.store.get_update_log(limit=50)

daily_dates = manager.store.get_table_trade_dates("stock_daily_raw")
basic_dates = manager.store.get_table_trade_dates("stock_daily_basic")
factor_dates = manager.store.get_table_trade_dates("stock_factor_pro")

print("stock_daily_raw 已入库交易日数:", len(daily_dates))
print("stock_daily_basic 已入库交易日数:", len(basic_dates))
print("stock_factor_pro 已入库交易日数:", len(factor_dates))

log_df.head(20)